In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
import re
from tqdm import tqdm
import random
import html
import unicodedata
import emoji
from bs4 import BeautifulSoup

output_path = 'C:\\Users\\shave\\OneDrive\\Desktop\\DSBA-6276-Hornets-Analysis\\data\\google_review_data.csv'

def scrape_google_reviews(place_url, num_reviews, max_runtime_minutes):
    """
    Scrape Google Maps reviews with improved reliability and scrolling
    """
    edge_driver_path = r'C:\Program Files (x86)\Microsoft\Edge\Application\msedgedriver.exe'
    
    # Set up Edge options
    edge_options = Options()
    edge_options.add_argument("--no-sandbox")
    edge_options.add_argument("--disable-dev-shm-usage")
    edge_options.add_argument("--disable-blink-features=AutomationControlled")
    edge_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
    
    # Initialize driver
    edge_service = Service(edge_driver_path)
    driver = webdriver.Edge(service=edge_service, options=edge_options)
    driver.set_page_load_timeout(60)
    
    # Data storage
    all_reviews = []
    seen_review_texts = set()  # Track seen reviews by text to avoid duplicates
    
    # Set time limits
    start_time = time.time()
    max_runtime_seconds = max_runtime_minutes * 60
    
    def save_progress():
        """Save current progress to file"""
        if all_reviews:
            df = pd.DataFrame(all_reviews)
            df.to_csv(output_path, index=False)
            print(f"[INFO] Saved {len(all_reviews)} reviews to {output_path}")
    
    def get_text(text):
        """Clean and normalize text"""
        if not text:
            return ""
        text = html.unescape(text)
        text = unicodedata.normalize('NFKC', text)
        text = emoji.replace_emoji(text, '')
        text = re.sub(r'[^\w\s.,;:!?\'"-]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
        
    def extract_review(review_element):
        """Extract data from a review element with BeautifulSoup"""
        try:
            try:
                more_buttons = review_element.find_elements(By.CSS_SELECTOR, 'button.w8nwRe, button[aria-label*="See more"]')
                if more_buttons and more_buttons[0].is_displayed():
                    driver.execute_script("arguments[0].click();", more_buttons[0])
                    time.sleep(0.2)
            except:
                pass
            
            html_content = review_element.get_attribute('outerHTML')
            soup = BeautifulSoup(html_content, 'html.parser')
            
            reviewer_element = soup.select_one('.d4r55, .WNxzHc, [class*="fontHead"]')
            reviewer_name = get_text(reviewer_element.text.strip()) if reviewer_element else "Anonymous"
            
            info_element = soup.select_one('.RfnDt, .g0dYBd, [class*="fontBody"]')
            reviewer_info = get_text(info_element.text.strip()) if info_element else ""
            
            rating_element = soup.select_one('span[aria-label*="star"], span[aria-label*="Stars"]')
            stars = "N/A"
            if rating_element and rating_element.has_attr('aria-label'):
                stars_text = rating_element['aria-label']
                stars_match = re.search(r'(\d+)', stars_text)
                stars = stars_match.group(1) if stars_match else "N/A"
            
            text_element = soup.select_one('.wiI7pd, .MyEned, [class*="review-full-text"], [data-review-id] div[style*="webkit-line-clamp"]')
            review_text = get_text(text_element.text.strip()) if text_element else "No text"
            
            date_element = soup.select_one('.rsqaWe, [class*="review-date"]')
            if date_element:
                date = get_text(date_element.text.strip())
            else:
                date_elements = [el for el in soup.find_all('span') if any(x in el.text for x in ['ago', 'month', 'week'])]
                date = get_text(date_elements[0].text.strip()) if date_elements else "N/A"
            
            return {
                'Reviewer': reviewer_name,
                'Reviewer Info': reviewer_info,
                'Stars': stars,
                'Review': review_text,
                'Date': date
            }
        except Exception as e:
            print(f"[ERROR] Failed to extract review: {str(e)}")
            return None

    try:
        # --- IMPLEMENTED FIXES START HERE ---
        # 1. Force a large desktop window size to reveal the Reviews tab
        driver.set_window_size(1920, 1080) 
        
        print(f"[INFO] Opening URL: {place_url}")
        driver.get(place_url)
        
        print("[INFO] Waiting 30 seconds for manual login or handling captchas...")
        time.sleep(30)
        
        # 2. Handle Cookie Consent (Issue #1)
        try:
            consent_xpath = "//button//span[contains(text(), 'Accept all')] | //button//span[contains(text(), 'I agree')]"
            consent_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, consent_xpath))
            )
            consent_button.click()
            print("[INFO] Cookie consent dismissed.")
            time.sleep(2)
        except:
            print("[INFO] No cookie popup detected.")

        driver.maximize_window()
        time.sleep(2)

        # 3. Aggressive & Robust Review Tab Click
        try:
            print("[INFO] Attempting to click the main 'Reviews' tab...")
            reviews_tab = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(@aria-label, 'Reviews for')] | //div[@role='tab']//div[contains(text(), 'Reviews')]"))
            )
            driver.execute_script("arguments[0].click();", reviews_tab)
            print("[INFO] Successfully navigated to Reviews via tab.")
            time.sleep(3)
        except:
            print("[WARNING] Primary Review Tab click failed. Using JS to find and click any review-related element...")
            try:
                # Injection of JS script to forcefully find the star rating or review tab and click it
                js_clicker = """
                var elements = document.querySelectorAll('button, div');
                var clicked = false;
                
                // 1st Priority: Look for any tab exactly containing "Reviews" text
                for(var i = 0; i < elements.length; i++) {
                    if (elements[i].getAttribute('role') === 'tab' && elements[i].textContent.includes('Reviews')) {
                        elements[i].click();
                        clicked = true;
                        break;
                    }
                }
                
                // 2nd Priority: Look for the star rating block right under the title
                if (!clicked) {
                    var stars = document.querySelectorAll('span[aria-label*="stars"], span[aria-label*="estrellas"]');
                    if (stars.length > 0) {
                        for (var i = 0; i < stars.length; i++) {
                            // Find the closest clickable parent (button or div)
                            var parent = stars[i].closest('button, div.F7nice, div[role="button"]');
                            if (parent) {
                                parent.click();
                                clicked = true;
                                break;
                            }
                        }
                    }
                }
                return clicked;
                """
                success = driver.execute_script(js_clicker)
                if success:
                    print("[INFO] Clicked the star rating or review tab via JavaScript.")
                    time.sleep(4)
                else:
                    print("[WARNING] Could not find any clickable review summary/tab. Will try scrolling the main pane...")
                    
                    # Scroll down the overview pane until a "More reviews" button appears
                    try:
                        scroll_pane = driver.find_element(By.CSS_SELECTOR, 'div[role="main"]')
                        for _ in range(5):
                            driver.execute_script("arguments[0].scrollTop += 800", scroll_pane)
                            time.sleep(1)
                            
                        # Try clicking "More reviews"
                        more_reviews = driver.find_elements(By.XPATH, "//button[.//span[contains(text(), 'More reviews')]] | //button[contains(@aria-label, 'More reviews')]")
                        if more_reviews:
                            driver.execute_script("arguments[0].click();", more_reviews[0])
                            print("[INFO] Clicked 'More reviews' button after scrolling.")
                            time.sleep(3)
                    except:
                        print("[WARNING] Scrolling attempt failed.")
                    
            except Exception as js_e:
                print(f"[ERROR] Auto-clicker encountered an issue: {str(js_e)}")
        # --- IMPLEMENTED FIXES END HERE ---

        # Scroll to load reviews
        print(f"[INFO] Starting to scroll and collect reviews...")
        prev_review_count = 0
        stall_count = 0
        review_container = None
        found_container = False
        
        with tqdm(total=num_reviews, desc="Collecting reviews") as pbar:
            # Find the reviews container
            try:
                review_container_selectors = [
                    'div.m6QErb.DxyBCb.kA9KIf.dS8AEf.XiKgde',
                    'div.m6QErb', 
                    'div.m6QErb.XiKgde',
                    'div[role="feed"]', 
                    'div.section-layout.section-scrollbox'
                ]
                
                for selector in review_container_selectors:
                    try:
                        review_container = driver.find_element(By.CSS_SELECTOR, selector)
                        if review_container.is_displayed():
                            print(f"[INFO] Found review container with selector: {selector}")
                            found_container = True
                            break
                    except:
                        continue
                
                if not found_container:
                    review_container = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.XPATH, '//div[contains(@class, "m6QErb") and .//div[contains(@class, "jftiEf")]]'))
                    )
                    print("[INFO] Found review container with fallback XPath")
                    found_container = True
            except Exception as e:
                print(f"[ERROR] Cannot find review container: {str(e)}")
                save_progress()
                return all_reviews
            
            if not found_container:
                print("[ERROR] Failed to locate review container after multiple attempts")
                save_progress()
                return all_reviews
            
            while len(all_reviews) < num_reviews:
                elapsed_time = time.time() - start_time
                if elapsed_time > max_runtime_seconds:
                    print(f"[INFO] Maximum runtime of {max_runtime_minutes} minutes reached")
                    break
                
                review_elements = review_container.find_elements(By.CSS_SELECTOR, 'div.jftiEf, div.gws-localreviews__google-review')
                
                if not review_elements:
                    stall_count += 1
                    if stall_count > 3:
                        break
                else:
                    new_reviews_found = False
                    for review_element in review_elements:
                        if len(all_reviews) >= num_reviews:
                            break
                        review = extract_review(review_element)
                        if review:
                            # Skip blank reviews
                            if not review.get('Review') or review['Review'].strip() in ['', 'No text', 'Sin texto']:
                                continue
                                
                            review_text_snippet = review['Review'][:50] if review['Review'] else ""
                            review_key = f"{review['Reviewer']}_{review['Stars']}_{review_text_snippet}"
                            
                            if review_key not in seen_review_texts:
                                all_reviews.append(review)
                                seen_review_texts.add(review_key)
                                pbar.update(1)
                                new_reviews_found = True
                
                if len(all_reviews) % 20 == 0 and len(all_reviews) > prev_review_count:
                    save_progress()
                
                if len(all_reviews) > prev_review_count or new_reviews_found:
                    prev_review_count = len(all_reviews)
                    stall_count = 0
                else:
                    stall_count += 1
                
                if stall_count > 5:
                    break
                
                driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", review_container)
                time.sleep(random.uniform(1.5, 3.0))
                pbar.set_description(f"Collected {len(all_reviews)} reviews")
                pbar.refresh()
        
        save_progress()
        return all_reviews
        
    except Exception as e:
        print(f"[ERROR] An error occurred: {str(e)}")
        driver.quit()   
        return all_reviews
    finally:
        driver.quit()

In [2]:
place_url = 'https://www.google.com/maps/place/Spectrum+Center/@35.2251952,-80.8419217,680m/data=!3m1!1e3!4m16!1m7!3m6!1s0x8856a0243e9c7d45:0x76230b8d88293efa!2sSpectrum+Center!8m2!3d35.2251952!4d-80.8393468!16zL20vMDI1eDN3!3m7!1s0x8856a0243e9c7d45:0x76230b8d88293efa!8m2!3d35.2251952!4d-80.8393468!9m1!1b1!16zL20vMDI1eDN3?entry=ttu&g_ep=EgoyMDI2MDMxNS4wIKXMDSoASAFQAw%3D%3D'

num_reviews = 500
max_runtime_minutes = 60

reviews_list = scrape_google_reviews(place_url, num_reviews, max_runtime_minutes)
reviews_df = pd.DataFrame(reviews_list)
reviews_df.to_csv(output_path, index=False)
print(f"[INFO] Reviews saved to: {output_path}")
print(reviews_df.head())

[INFO] Opening URL: https://www.google.com/maps/place/Spectrum+Center/@35.2251952,-80.8419217,680m/data=!3m1!1e3!4m16!1m7!3m6!1s0x8856a0243e9c7d45:0x76230b8d88293efa!2sSpectrum+Center!8m2!3d35.2251952!4d-80.8393468!16zL20vMDI1eDN3!3m7!1s0x8856a0243e9c7d45:0x76230b8d88293efa!8m2!3d35.2251952!4d-80.8393468!9m1!1b1!16zL20vMDI1eDN3?entry=ttu&g_ep=EgoyMDI2MDMxNS4wIKXMDSoASAFQAw%3D%3D
[INFO] Waiting 30 seconds for manual login or handling captchas...
[INFO] No cookie popup detected.
[INFO] Attempting to click the main 'Reviews' tab...
[INFO] Successfully navigated to Reviews via tab.
[INFO] Starting to scroll and collect reviews...


[INFO] Found review container with selector: div.m6QErb.DxyBCb.kA9KIf.dS8AEf.XiKgde


Collected 113 reviews:  24%|██▍       | 120/500 [01:17<04:15,  1.49it/s]

[INFO] Saved 120 reviews to C:\Users\shave\OneDrive\Desktop\DSBA-6276-Hornets-Analysis\data\google_review_data.csv


Collected 196 reviews:  40%|████      | 200/500 [03:08<05:47,  1.16s/it]

...

[INFO] Saved 380 reviews to C:\Users\shave\OneDrive\Desktop\DSBA-6276-Hornets-Analysis\data\google_review_data.csv


Collected 495 reviews: 100%|█████████▉| 498/500 [15:30<00:05,  2.88s/it]

[INFO] Saved 500 reviews to C:\Users\shave\OneDrive\Desktop\DSBA-6276-Hornets-Analysis\data\google_review_data.csv


Collected 500 reviews: 100%|██████████| 500/500 [15:33<00:00,  1.87s/it]


[INFO] Saved 500 reviews to C:\Users\shave\OneDrive\Desktop\DSBA-6276-Hornets-Analysis\data\google_review_data.csv
[INFO] Reviews saved to: C:\Users\shave\OneDrive\Desktop\DSBA-6276-Hornets-Analysis\data\google_review_data.csv
                                           Reviewer  \
0  Clarence StewartLocal Guide 108 reviews 2 photos   
1        NNeo NeoLocal Guide 160 reviews 139 photos   
2                        j hodges4 reviews 2 photos   
3                     Adrian Abud8 reviews 4 photos   
4                                 Sue Hill7 reviews   

                                       Reviewer Info Stars  \
0  Clarence StewartLocal Guide 108 reviews 2 phot...     5   
1  NNeo NeoLocal Guide 160 reviews 139 photos3 da...     5   
2  j hodges4 reviews 2 photos3 days ago New Seat ...     5   
3  Adrian Abud8 reviews 4 photos3 days ago New Gr...     5   
4  Sue Hill7 reviews3 days ago New Prior to atten...     2   

                                              Review         Date  
0

### Clean & Format the Scraped Data
The cell below takes the raw CSV data and parses out the "Local Guide" status, the number of reviews, and the number of photos into their own isolated columns.

In [3]:
import pandas as pd
import re

def clean_scraped_data(output_path):
    # Load the raw scraped data
    df = pd.read_csv(output_path)
    
    cleaned_rows = []
    
    for _, row in df.iterrows():
        # Combine Reviewer name and Reviewer Info variables to catch all metadata
        raw_reviewer = str(row.get('Reviewer', ''))
        raw_info = str(row.get('Reviewer Info', ''))
        combined_string = raw_reviewer + " " + raw_info
        
        # 1. Check for 'Local Guide'
        local_guide = "TRUE" if "Local Guide" in combined_string else "FALSE"
        
        # 2. Extract Reviews count
        reviews_match = re.search(r'([\d,]+)\s*review', combined_string, flags=re.IGNORECASE)
        if reviews_match:
            reviews_count = int(reviews_match.group(1).replace(',', ''))
        else:
            reviews_count = 0
            
        # 3. Extract Photos count
        photos_match = re.search(r'([\d,]+)\s*photo', combined_string, flags=re.IGNORECASE)
        if photos_match:
            photos_count = int(photos_match.group(1).replace(',', ''))
        else:
            photos_count = 0
            
        # 4. Clean the Reviewer's Name to leave only the username
        name = raw_reviewer
        name = re.sub(r'Local\s*Guide.*', '', name, flags=re.IGNORECASE)
        name = re.sub(r'[\d,]+\s*reviews?.*', '', name, flags=re.IGNORECASE)
        name = re.sub(r'[\d,]+\s*photos?.*', '', name, flags=re.IGNORECASE)
        name = name.strip()
        
        # Build the structured row mapping to your exact requested format
        cleaned_rows.append({
            'Reviewer': name,
            'Local Guide': local_guide,
            'Reviews': reviews_count,
            'Photos': photos_count,
            'Stars': row.get('Stars', ''),
            'Review': row.get('Review', ''),
            'Date': row.get('Date', '')
        })
        
    return pd.DataFrame(cleaned_rows)

# Execute the cleaning function
cleaned_data = clean_scraped_data(output_path)

# Display the newly formatted data layout
display(cleaned_data.head())

# Save it to a new file so the original raw file remains safe
cleaned_file_path = output_path
cleaned_data.to_csv(cleaned_file_path, index=False)
print(f"[INFO] Cleaned tabular data saved to: {cleaned_file_path}")

,Reviewer,Local Guide,Reviews,Photos,Stars,Review,Date
0,Clarence Stewart,TRUE,108,2,5,New Editions Tour awesome time with daughter f...,an hour ago
1,NNeo Neo,TRUE,160,139,5,The staff is always helpful here at spectrum c...,3 days ago
2,j hodges,FALSE,4,2,5,Seat views are much closer in person than they...,3 days ago
3,Adrian Abud,FALSE,8,4,5,Great time,3 days ago
4,Sue Hill,FALSE,7,0,2,"Prior to attending the ACC tournament, I email...",3 days ago


[INFO] Cleaned tabular data saved to: C:\Users\shave\OneDrive\Desktop\DSBA-6276-Hornets-Analysis\data\google_review_data.csv


In [ ]:
import pandas as pd
import re

# Load the file produced by the cell above
df_final = pd.read_csv(output_path)

def extract_review_metadata(text):
    text = str(text) if pd.notna(text) else ""
    visited_on = ""
    wait_time = ""
    reservation = ""
    
    # Try to find and extract the 3 specific metadata fields
    # Use non-greedy matching until the next keyword or end of string
    v_match = re.search(r'Visited on\s*([A-Za-z0-9\s\-\+–]+?)(?=Wait time|Reservation recommended|$)', text)
    w_match = re.search(r'Wait time\s*([A-Za-z0-9\s\-\+–]+?)(?=Visited on|Reservation recommended|$)', text)
    r_match = re.search(r'Reservation recommended\s*([A-Za-z0-9\s\-\+–]+?)(?=Visited on|Wait time|$)', text)
    
    if v_match:
        visited_on = v_match.group(1).strip()
        text = text.replace(v_match.group(0), '')
    if w_match:
        wait_time = w_match.group(1).strip()
        text = text.replace(w_match.group(0), '')
    if r_match:
        reservation = r_match.group(1).strip()
        text = text.replace(r_match.group(0), '')
        
    return text.strip(), visited_on, wait_time, reservation

# Apply the parsing logically against the Review column, creating 4 outputs (1 updated, 3 new columns)
df_final[['Review', 'Visited on', 'Wait time', 'Reservation recommended']] = df_final['Review'].apply(
    lambda x: pd.Series(extract_review_metadata(x))
)

# Display the cleanly formatted layout
display(df_final.head())

# Save it to complete the pipeline
advanced_clean_path = output_path
df_final.to_csv(advanced_clean_path, index=False)
print(f"[INFO] New columns successfully added. Data saved to: {advanced_clean_path}")